# Gemini sibling specialist — isolated OOF diagnostic

This experiment does not modify production and does not submit to Kaggle. It measures whether a 3-way Gemini specialist improves the same global char+word SVC under 5-fold `GroupKFold` by `prompt_id`. The specialist may only change predictions already gated into the Gemini family; it can never move a non-Gemini prediction into Gemini. Selection requires a predeclared mean balanced-accuracy gain of at least +0.010 over the same-fold baseline.

In [ ]:
import glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score

paths = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
if not paths:
    raise FileNotFoundError("train.csv not found under /kaggle/input")
train = pd.read_csv(paths[0])
required = {"id", "prompt_id", "text", "model"}
assert required <= set(train.columns), (required, train.columns.tolist())
train["text"] = train["text"].fillna("").astype(str)
labels = sorted(train["model"].astype(str).unique())
assert len(labels) == 18, labels
label_to_i = {x:i for i,x in enumerate(labels)}
y = train["model"].map(label_to_i).to_numpy()
groups = train["prompt_id"].astype(str).to_numpy()
GEMINI = ["gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash"]
SIBLINGS = ["gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash"]
gemini_idx = np.array([label_to_i[x] for x in GEMINI])
sibling_idx = np.array([label_to_i[x] for x in SIBLINGS])
print({"rows": len(train), "prompt_groups": len(np.unique(groups)), "labels": labels}, flush=True)


In [ ]:
def make_features(tr_text, va_text):
    tr_blocks, va_blocks = [], []
    specs = [("char", (3,5), 180000), ("word", (1,3), 140000)]
    for analyzer, ngram_range, max_features in specs:
        vec = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, sublinear_tf=True,
            min_df=2, max_features=max_features, lowercase=False,
            dtype=np.float32
        )
        tr_blocks.append(vec.fit_transform(tr_text))
        va_blocks.append(vec.transform(va_text))
    return hstack(tr_blocks, format="csr"), hstack(va_blocks, format="csr")

def softmax_like(dec):
    z = dec - dec.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)


In [ ]:
# Predeclared threshold grid. The specialist only replaces rows whose global top-1 is Gemini.
THRESHOLDS = [0.00, 0.10, 0.20, 0.35, 0.50, 0.70]
fold_records = []
oof_base = np.full(len(train), -1, dtype=int)
oof_by_threshold = {t: np.full(len(train), -1, dtype=int) for t in THRESHOLDS}
t0 = time.time()
gkf = GroupKFold(n_splits=5)
for fold, (itr, iva) in enumerate(gkf.split(train, y, groups), 1):
    # Strict group check: no prompt_id crosses train and validation.
    assert set(groups[itr]).isdisjoint(set(groups[iva]))
    Xtr, Xva = make_features(train.iloc[itr].text.tolist(), train.iloc[iva].text.tolist())
    global_model = LinearSVC(C=1.5, class_weight="balanced", max_iter=5000, random_state=100+fold)
    global_model.fit(Xtr, y[itr])
    dec = global_model.decision_function(Xva)
    base_pred = dec.argmax(axis=1)
    order = np.argsort(dec, axis=1)
    top = order[:, -1]
    second = order[:, -2]
    margin = dec[np.arange(len(iva)), top] - dec[np.arange(len(iva)), second]
    # Specialist is trained only on Gemini sibling rows from the training fold.
    train_sib = np.isin(y[itr], sibling_idx)
    assert train_sib.any()
    specialist = LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=200+fold)
    specialist.fit(Xtr[train_sib], y[itr][train_sib])
    spec_pred = specialist.predict(Xva)
    oof_base[iva] = base_pred
    row_stats = {"fold": fold, "rows": int(len(iva)), "base_bal_acc": float(balanced_accuracy_score(y[iva], base_pred))}
    for threshold in THRESHOLDS:
        # Family gate: never alter a non-Gemini top-1 prediction.
        gate = np.isin(top, gemini_idx) & (margin <= threshold if threshold > 0 else np.zeros(len(iva), dtype=bool))
        # At threshold 0, measure the ungated control exactly: no replacements.
        pred = base_pred.copy()
        pred[gate] = spec_pred[gate]
        oof_by_threshold[threshold][iva] = pred
        row_stats[f"t_{threshold:.2f}"] = {
            "balanced_accuracy": float(balanced_accuracy_score(y[iva], pred)),
            "changed_rows": int(gate.sum()),
            "gemini_gate_rows": int(np.isin(top, gemini_idx).sum()),
        }
    fold_records.append(row_stats)
    print(row_stats, flush=True)


In [ ]:
summary = []
for threshold in THRESHOLDS:
    pred = oof_by_threshold[threshold]
    score = balanced_accuracy_score(y, pred)
    recalls = recall_score(y, pred, labels=np.arange(len(labels)), average=None, zero_division=0)
    gemini_recall = float(np.mean([recalls[label_to_i[x]] for x in SIBLINGS]))
    summary.append({
        "threshold": threshold,
        "balanced_accuracy": float(score),
        "delta_vs_no_specialist": float(score - balanced_accuracy_score(y, oof_base)),
        "gemini_sibling_macro_recall": gemini_recall,
        "changed_rows": int(np.sum(pred != oof_base)),
        "per_class_recall": {labels[i]: float(recalls[i]) for i in range(len(labels))},
    })
base_score = float(balanced_accuracy_score(y, oof_base))
positive = [r for r in summary if r["delta_vs_no_specialist"] >= 0.010]
selected = max(positive, key=lambda r:r["balanced_accuracy"]) if positive else None
result = {
    "experiment": "gemini_3way_sibling_specialist",
    "protocol": "5-fold GroupKFold by prompt_id",
    "rows": int(len(train)),
    "prompt_groups": int(len(np.unique(groups))),
    "base_balanced_accuracy": base_score,
    "predeclared_promotion_margin": 0.010,
    "thresholds": summary,
    "selected": selected,
    "folds": fold_records,
    "elapsed_seconds": round(time.time()-t0, 1),
}
Path("/kaggle/working/gemini_sibling_results.json").write_text(json.dumps(result, indent=2))
print(json.dumps({k:result[k] for k in ["experiment","protocol","base_balanced_accuracy","selected","elapsed_seconds"]}, indent=2), flush=True)
